In [0]:
# ===========================================================================
# Notebook  : bronze_account
# Location  : /HGI-Lakehouse-Pipeline/02_Bronze_Layer/bronze_account
# Purpose   : SALESFORCE Account → bronze.{customer_id}.crm_accounts
#
# Serverless : Set max_files_per_trigger=10 for Serverless 2X-Small testing
# Job Compute: Set max_files_per_trigger=500 for production
# ===========================================================================



In [0]:
# CELL 1 ── Load shared config
# In Databricks: uncomment the two %run lines below.
# Each %run MUST be in its own cell (magic commands require their own cell).
%run ../config/pipeline_config.py

In [0]:
%run ./bronze_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id",           "")
dbutils.widgets.text("load_type",             "")   # historical | incremental
dbutils.widgets.text("max_files_per_trigger", "500")
# Serverless tip: override max_files_per_trigger to '10' on Serverless 2X-Small

customer_id           = dbutils.widgets.get("customer_id").strip().lower()
load_type             = dbutils.widgets.get("load_type").strip().lower()
max_files_per_trigger = int(dbutils.widgets.get("max_files_per_trigger"))

In [0]:
# CELL 3 ── Object-specific constants
source_system  = "salesforce"
object_name    = "account"
sf_object_name = "Account"
# Composite ID will be: salesforce_Account_Id_{SalesforceId}
tenant_id = tenant_id_from_customer(customer_id)   # helper from pipeline_config

In [0]:
# Ensure audit tables exist for error logging
initialize_audit_tables()

In [0]:
# CELL 4 -- Path resolution  (all helpers from pipeline_config)
landing_zone_path = landing_path(source_system, customer_id, object_name, load_type)
checkpoint_loc    = checkpoint_path("bronze", source_system, customer_id, object_name) + load_type + "/"  # Bug 1 fix: separate checkpoint per load_type
schema_loc        = checkpoint_loc + "_schema/"
table_full_name   = bronze_table(customer_id, object_name)
# table_full_name = bronze.{customer_id}.crm_accounts

print(f"=== Bronze: SALESFORCE Account ===")
print(f"  Customer   : {customer_id}  (tenant={tenant_id})")
print(f"  Load type  : {load_type}")
print(f"  Landing    : {landing_zone_path}")
print(f"  Checkpoint : {checkpoint_loc}")
print(f"  Target     : {table_full_name}")
print(f"  Batch size : {max_files_per_trigger} files/trigger")

In [0]:
# CELL 5 ── Create bronze schema + table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}.{customer_id}")
create_bronze_table(table_full_name, object_name)  # uses BRONZE_DDL_COLUMNS['account']

In [0]:
# CELL 6 ── Run pipeline
# Pre-check: skip gracefully if the landing zone path doesn't exist yet
try:
    dbutils.fs.ls(landing_zone_path)
except Exception:
    print(f"⚠️ No data at landing zone path: {landing_zone_path}")
    print("Skipping ingestion — no files to process.")
    dbutils.notebook.exit("NO_DATA")

try:
    # start_ingestion() returns the bronze_logger summary dict
    load_summary = run_with_retry(start_ingestion)

    # Log success with record counts from the logger
    log_audit(
        job_name       = f"bronze_{object_name}",
        customer_id    = customer_id,
        status         = "success",
        layer          = "bronze",
        alert_type     = "SUCCESS",
        rows_processed = bronze_logger.total_merged,
    )
except Exception as e:
    print(f"❌ Pipeline failed: {e}")
    log_audit(
        job_name       = f"bronze_{object_name}",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "bronze",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise

In [0]:
# CELL 7 ── Conditional Post-load maintenance
# FIX: Only run OPTIMIZE for large batches (>1000 rows), skip VACUUM on incremental runs
OPTIMIZE_THRESHOLD = 1000  # Only OPTIMIZE if more than this many rows were merged

try:
    # Get the count of rows merged in this run from bronze_logger
    rows_merged_this_run = bronze_logger.total_merged if hasattr(bronze_logger, 'total_merged') else 0
    
    if rows_merged_this_run > OPTIMIZE_THRESHOLD:
        print(f"  Running OPTIMIZE ({rows_merged_this_run:,} rows merged > {OPTIMIZE_THRESHOLD} threshold)...")
        spark.sql(f"OPTIMIZE {table_full_name}")
        print(f"  OPTIMIZE complete")
    else:
        print(f"  Skipping OPTIMIZE ({rows_merged_this_run:,} rows merged <= {OPTIMIZE_THRESHOLD} threshold)")
    
    # VACUUM only on historical loads, not incremental (saves ~30-60s per run)
    if load_type == "historical":
        print(f"  Running VACUUM (historical load)...")
        spark.sql(f"VACUUM {table_full_name} RETAIN 168 HOURS")
        print(f"  VACUUM complete")
    else:
        print(f"  Skipping VACUUM (incremental load — schedule weekly instead)")
    
    print(f"Bronze load complete: {table_full_name}")
    
except Exception as e:
    print(f"⚠️ Post-load maintenance warning (non-fatal): {e}")
    # Don't fail the job for OPTIMIZE/VACUUM issues

In [0]:
total_rows = spark.table(table_full_name).count()
print(f"\n✅ Bronze load complete: {total_rows:,} total rows in {table_full_name}")
print(f"   CDC records merged this run: {bronze_logger.total_merged:,}")
print(f"   Records from landing: {bronze_logger.total_landing:,}")
print(f"   Duplicates removed: {bronze_logger.total_landing - bronze_logger.total_deduped:,}")
print(f"   Unchanged skipped: {bronze_logger.total_deduped - bronze_logger.total_merged:,}")